# Generate the reusable matched MVTec split

This notebook creates one deterministic, category-balanced and label-balanced 50/50 manifest. Upload the resulting `splits/` directory as a Kaggle Dataset and reuse it in every adversarial run.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE/UPDATE REPOSITORY =====')
working = Path('/kaggle/working')
repo_root = working / 'adversarial-robustness'
repo_url = 'https://github.com/Parsagh05/adversarial-robustness.git'
if repo_root.exists():
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_root / 'requirements.txt')],
    check=True,
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Repository ready:', repo_root)

In [ ]:
import json
from adversarial_harness import create_matched_split_manifest

print('===== STEP 2: GENERATE MATCHED SPLIT =====')
MVTEC_ROOT = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
SPLIT_SEED = 111
FIT_FRACTION = 0.5
CATEGORIES = None  # None means every discovered MVTec category.

if not MVTEC_ROOT.is_dir():
    raise FileNotFoundError(f'MVTec mount not found: {MVTEC_ROOT}')

split_dir = Path('/kaggle/working/splits')
csv_path = split_dir / 'mvtec_matched_test_per_category_v1_seed111.csv'
json_path = split_dir / 'mvtec_matched_test_per_category_v1_seed111.json'
create_matched_split_manifest(
    mvtec_root=str(MVTEC_ROOT),
    csv_path=str(csv_path),
    json_path=str(json_path),
    categories=CATEGORIES,
    split_seed=SPLIT_SEED,
    fit_fraction=FIT_FRACTION,
)
metadata = json.loads(json_path.read_text(encoding='utf-8'))
print('CSV:', csv_path)
print('JSON:', json_path)
print('CSV SHA256:', metadata['csv_sha256'])
print('Selected rows:', metadata['total_selected_rows'])
for category, counts in metadata['category_counts'].items():
    print(category, counts)

In [ ]:
import shutil

print('===== STEP 3: PACKAGE OUTPUTS =====')
archive = shutil.make_archive(
    '/kaggle/working/mvtec_matched_test_per_category_v1_seed111',
    'zip',
    root_dir=split_dir.parent,
    base_dir=split_dir.name,
)
print('Packaged split manifest:', archive)